# P03 · Controls (RQ4) + ImageNet baseline (RQ1), development events

Asks two questions of benchmark v0 (p02/p02b), on the **development and secondary** events only:

1. **How much of the score is the building itself?** Two controls go through the same probes:
   * **image-free:** building size and how crowded its surroundings are, no image at all;
   * **neighbourhood (ring 10 m):** the encoder's tokens within 10 m of the building, **excluding every token that
     contains part of the building**, so the probe sees the surroundings but not the building.
   If a control reaches the encoder's AUROC, the encoder is not recognising damage to the building.
2. **Does DINOv3 beat an ImageNet-supervised ViT-B?** (preview of RQ1 on development events).

* Reuses p02's DINOv3 outline/dilated features (native resolution, ADR 0001). Extracts DINOv3 **ring** features
  and ImageNet ViT-B/16 outline, dilated 3 m and ring features at native resolution.
* **Exploratory only.** Confirmatory tiles are skipped; every verdict is "exploratory reading, not a claim".
* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN`.
  Needs `01_data/derived/buildings.parquet` (p01) and `02_models/features/2026-11-20_dinov3_resolution/` (p02).
* **Time:** about 45 minutes (data ~5 min, DINOv3 ring ~10 min, ImageNet ~12 min, bench run ~15 min).
* **Outputs:** features in `02_models/features/<date>_controls_imagenet/`, the run in this run's folder.

In [ ]:
NOTEBOOK_NAME = "p03_controls_and_imagenet_v0"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU and Hugging Face token (DINOv3 is gated; ImageNet ViT-B is public)
import os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
P02_FEATS = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
assert os.path.exists(P02_FEATS), f"missing {P02_FEATS}: run p02 first"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import datetime, gc
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_controls_imagenet"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# Control 1: image-free (building size and crowding; no pixels)
from damagesar_bench.controls.imagefree import image_free_features
from damagesar_bench.encoders.extract import write_features
free = image_free_features(use, events)
FREE = write_features(free, f"{FEAT_DIR}/imagefree.parquet")
print(len(free), "buildings ->", FREE)
del free; gc.collect()

In [ ]:
# DINOv3 ring features (native): control 2 for DINOv3. Outline/dilated come from p02.
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
RING = "ring:10"
enc = get_encoder("dinov3_vitb16")
f, runs_d = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, [RING], [None],
                                   "/content/features/dinov3_ring", events, log=print)
DINO_RING = write_features(f, f"{FEAT_DIR}/dinov3_ring.parquet")
ring_fallback_dino = f.groupby("event")["pool_pixels"].apply(lambda s: float((s == 0).mean())).rename("dinov3")
del f, enc; gc.collect(); torch.cuda.empty_cache()
display(runs_table(runs_d))

In [ ]:
# ImageNet ViT-B/16 (supervised; RQ1 baseline) at native resolution: outline, dilated 3 m, ring 10 m
enc = get_encoder("imagenet_vitb16")
f, runs_i = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline", "dilated:3", RING],
                                   [None], "/content/features/imagenet", events, log=print)
IMNET = write_features(f, f"{FEAT_DIR}/imagenet.parquet")
ring_fallback_imnet = f[f["pooling"] == "ring10m"].groupby("event")["pool_pixels"].apply(
    lambda s: float((s == 0).mean())).rename("imagenet")
del f, enc; gc.collect(); torch.cuda.empty_cache()
timing = pd.concat([runs_table(runs_d), runs_table(runs_i)], ignore_index=True)
timing.to_csv(f"{RUN}/timing.csv", index=False); timing.to_csv(f"{FEAT_DIR}/timing.csv", index=False)
display(timing)
fallback = pd.concat([ring_fallback_dino, ring_fallback_imnet], axis=1)
fallback.to_csv(f"{RUN}/ring_fallback_share.csv")
print("Share of buildings whose ring fell back to the 8 neighbour tokens:"); print(fallback.round(3).to_string())

In [ ]:
# bench run: only the settings the comparisons use (p02's 0.3 m settings are skipped)
del use; gc.collect()
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
D, I = "dinov3_vitb16@native", "imagenet_vitb16@native"
comps = [
    ComparisonSpec((D, "outline"), (I, "outline"), "DINOv3 vs ImageNet (outline)"),
    ComparisonSpec((D, "outline"), (D, "ring10m"), "building vs surroundings (DINOv3)"),
    ComparisonSpec((I, "outline"), (I, "ring10m"), "building vs surroundings (ImageNet)"),
    ComparisonSpec((D, "outline"), ("imagefree", "geometry"), "DINOv3 vs image-free"),
    ComparisonSpec((I, "outline"), ("imagefree", "geometry"), "ImageNet vs image-free"),
    ComparisonSpec((I, "dilated3m"), (I, "outline"), "dilated3m vs outline (ImageNet)"),
]
out = run_benchmark([P02_FEATS, str(DINO_RING), str(IMNET), str(FREE)], comps, events, f"{RUN}/bench_run",
                    n_boot=2000, only_compared=True)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))

In [ ]:
# Compact view: AUROC per event and setting; comparisons; spread of the gains
import json, numpy as np
m = pd.read_csv(f"{out}/event_metrics.csv")
pv = m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc")
pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string())
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "n_events", "estimate", "lower", "upper", "verdict"]].round(4).to_string(index=False))
for _, r in c.iterrows():
    g = np.array(list(json.loads(r["gains"]).values()))
    print(f"{r['label']}: per-event gains {json.loads(r['gains'])} | SD {g.std(ddof=1):.4f}")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))